# 03 — Graph Metric Functions

Core functions for consensus community detection and computation of **modularity** and **participation coefficient**.

In [1]:
import warnings
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('whitegrid')

try:
    import os
    import yaml
    from pathlib import Path
    config_name = os.environ.get('GRAPH_THEORY_CONFIG', 'config.yaml')
    with open(Path('..') / config_name, encoding='utf-8') as f:
        graph_cfg = yaml.safe_load(f)['graph_theory']
    CONSENSUS_RUNS = int(graph_cfg.get('consensus_runs', 100))
    CONSENSUS_THRESHOLD = float(graph_cfg.get('consensus_threshold', 0.5))
except FileNotFoundError:
    CONSENSUS_RUNS = 100
    CONSENSUS_THRESHOLD = 0.5

print(f'consensus_runs={CONSENSUS_RUNS}, consensus_threshold={CONSENSUS_THRESHOLD}')


consensus_runs=20, consensus_threshold=0.5


In [2]:
def safe_community_detection(G, n_runs=100, consensus_threshold=0.5):
    """使用共识聚类（consensus clustering）进行社区检测，获得稳定的划分
    consensus_threshold: 共现频率阈值，过滤掉低于该值的弱连接（默认0.5）
    """
    nodes = list(G.nodes())
    n_nodes = len(nodes)

    if n_nodes == 0:
        return {}, np.nan

    cooccurrence_matrix = np.zeros((n_nodes, n_nodes))

    try:
        import community

        all_partitions = []
        for _ in range(n_runs):
            partition = community.best_partition(G)
            all_partitions.append(partition)

        node_to_idx = {node: i for i, node in enumerate(nodes)}
        for partition in all_partitions:
            node_communities = {node: partition[node] for node in nodes}
            for i, node_i in enumerate(nodes):
                for j, node_j in enumerate(nodes):
                    if i <= j:
                        if node_communities[node_i] == node_communities[node_j]:
                            cooccurrence_matrix[i, j] += 1
                            if i != j:
                                cooccurrence_matrix[j, i] += 1

        cooccurrence_matrix = cooccurrence_matrix / n_runs

        consensus_G = nx.Graph()
        consensus_G.add_nodes_from(nodes)
        for i, node_i in enumerate(nodes):
            for j, node_j in enumerate(nodes):
                if i < j and cooccurrence_matrix[i, j] >= consensus_threshold:
                    consensus_G.add_edge(node_i, node_j, weight=cooccurrence_matrix[i, j])

        if consensus_G.number_of_edges() == 0:
            return {node: idx for idx, node in enumerate(nodes)}, np.nan

        final_partition = community.best_partition(consensus_G, weight="weight")
        final_modularity = community.modularity(final_partition, consensus_G, weight="weight")
        return final_partition, final_modularity

    except ImportError:
        try:
            from networkx.algorithms import community

            all_partitions = []
            for _ in range(n_runs):
                partition = community.greedy_modularity_communities(G)
                node_to_community = {}
                for i, comm in enumerate(partition):
                    for node in comm:
                        node_to_community[node] = i
                all_partitions.append(node_to_community)

            for partition in all_partitions:
                node_communities = {node: partition[node] for node in nodes}
                for i, node_i in enumerate(nodes):
                    for j, node_j in enumerate(nodes):
                        if i <= j:
                            if node_communities[node_i] == node_communities[node_j]:
                                cooccurrence_matrix[i, j] += 1
                                if i != j:
                                    cooccurrence_matrix[j, i] += 1

            cooccurrence_matrix = cooccurrence_matrix / n_runs

            consensus_G = nx.Graph()
            consensus_G.add_nodes_from(nodes)
            for i, node_i in enumerate(nodes):
                for j, node_j in enumerate(nodes):
                    if i < j and cooccurrence_matrix[i, j] >= consensus_threshold:
                        consensus_G.add_edge(node_i, node_j, weight=cooccurrence_matrix[i, j])

            if consensus_G.number_of_edges() == 0:
                return {node: idx for idx, node in enumerate(nodes)}, np.nan

            final_partition_sets = community.greedy_modularity_communities(
                consensus_G, weight="weight"
            )
            final_partition = {}
            for i, comm in enumerate(final_partition_sets):
                for node in comm:
                    final_partition[node] = i

            final_modularity = community.modularity(
                consensus_G, final_partition_sets, weight="weight"
            )
            return final_partition, final_modularity

        except Exception:
            return {node: 0 for node in G.nodes()}, np.nan


In [3]:
def calculate_graph_metrics(adjacency_matrix, threshold, metric_type="global", return_node_level=False):
    """简化版本的图论指标计算函数，仅保留模块度相关指标"""
    if np.any(np.isnan(adjacency_matrix)):
        if np.all(np.isnan(adjacency_matrix)):
            result = {
                f"{metric_type}_modularity": np.nan,
                f"{metric_type}_participation_coefficient": np.nan,
                f"{metric_type}_within_module_degree": np.nan,
                f"{metric_type}_num_modules": np.nan,
                f"{metric_type}_participation_entropy": np.nan,
            }
            if return_node_level:
                n_nodes = len(adjacency_matrix)
                result["node_level"] = {
                    "participation_coefficient": np.full(n_nodes, np.nan),
                    "within_module_degree": np.full(n_nodes, np.nan),
                    "participation_entropy": np.full(n_nodes, np.nan),
                }
            return result
        adjacency_matrix = np.nan_to_num(adjacency_matrix, nan=0.0)

    n_nodes_original = len(adjacency_matrix)
    upper_tri = np.triu(adjacency_matrix, k=1)
    valid_connections = upper_tri[~np.isnan(upper_tri) & (upper_tri > 0)]
    total_possible_connections = len(valid_connections)

    if total_possible_connections == 0:
        result = {
            f"{metric_type}_modularity": np.nan,
            f"{metric_type}_participation_coefficient": np.nan,
            f"{metric_type}_within_module_degree": np.nan,
            f"{metric_type}_num_modules": np.nan,
            f"{metric_type}_participation_entropy": np.nan,
        }
        if return_node_level:
            result["node_level"] = {
                "participation_coefficient": np.full(n_nodes_original, np.nan),
                "within_module_degree": np.full(n_nodes_original, np.nan),
                "participation_entropy": np.full(n_nodes_original, np.nan),
            }
        return result

    n_connections_to_keep = int(total_possible_connections * threshold)

    if n_connections_to_keep > 0:
        valid_mask = ~np.isnan(upper_tri) & (upper_tri > 0)
        positive_connections = upper_tri[valid_mask]
        positive_indices = np.where(valid_mask)
        sorted_indices = np.argsort(positive_connections)[::-1]
        keep_indices = sorted_indices[:n_connections_to_keep]

        binary_adj = np.zeros_like(adjacency_matrix)
        for idx in keep_indices:
            i, j = positive_indices[0][idx], positive_indices[1][idx]
            binary_adj[i, j] = 1
            binary_adj[j, i] = 1
    else:
        binary_adj = np.zeros_like(adjacency_matrix)

    G = nx.from_numpy_array(binary_adj)
    node_mapping = None
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        node_mapping = {node: node for node in largest_cc}
        G = G.subgraph(largest_cc).copy()
        if len(G.nodes()) < 3:
            result = {
                f"{metric_type}_modularity": np.nan,
                f"{metric_type}_participation_coefficient": np.nan,
                f"{metric_type}_within_module_degree": np.nan,
                f"{metric_type}_num_modules": np.nan,
                f"{metric_type}_participation_entropy": np.nan,
            }
            if return_node_level:
                result["node_level"] = {
                    "participation_coefficient": np.full(n_nodes_original, np.nan),
                    "within_module_degree": np.full(n_nodes_original, np.nan),
                    "participation_entropy": np.full(n_nodes_original, np.nan),
                }
            return result

    metrics = {}
    node_participation_coeffs = np.full(n_nodes_original, np.nan)
    node_within_module_degrees = np.full(n_nodes_original, np.nan)
    node_participation_entropies = np.full(n_nodes_original, np.nan)

    partition, modularity = safe_community_detection(G, n_runs=100, consensus_threshold=0.5)
    metrics[f"{metric_type}_modularity"] = modularity

    if len(set(partition.values())) > 1:
        participation_coeffs = []
        for node in G.nodes():
            node_community = partition[node]
            node_degree = G.degree(node)
            if node_degree > 0:
                within_community_degree = sum(
                    1 for neighbor in G.neighbors(node) if partition[neighbor] == node_community
                )
                participation_coeff = 1 - (within_community_degree / node_degree) ** 2
                participation_coeffs.append(participation_coeff)
                if return_node_level:
                    node_participation_coeffs[node] = participation_coeff

        if participation_coeffs:
            metrics[f"{metric_type}_participation_coefficient"] = np.mean(participation_coeffs)
        else:
            metrics[f"{metric_type}_participation_coefficient"] = np.nan
    else:
        metrics[f"{metric_type}_participation_coefficient"] = np.nan

    if len(set(partition.values())) > 1:
        within_module_degrees = []
        for node in G.nodes():
            node_community = partition[node]
            within_degree = sum(
                1 for neighbor in G.neighbors(node) if partition[neighbor] == node_community
            )
            community_nodes = [n for n in G.nodes() if partition[n] == node_community]
            community_degrees = [G.degree(n) for n in community_nodes]
            if len(community_degrees) > 1:
                mean_degree = np.mean(community_degrees)
                std_degree = np.std(community_degrees, ddof=1)
                if std_degree > 0:
                    z_score = (within_degree - mean_degree) / std_degree
                    within_module_degrees.append(z_score)
                    if return_node_level:
                        node_within_module_degrees[node] = z_score

        if within_module_degrees:
            metrics[f"{metric_type}_within_module_degree"] = np.mean(within_module_degrees)
        else:
            metrics[f"{metric_type}_within_module_degree"] = np.nan
    else:
        metrics[f"{metric_type}_within_module_degree"] = np.nan

    metrics[f"{metric_type}_num_modules"] = len(set(partition.values()))

    if len(set(partition.values())) > 1:
        participation_entropies = []
        for node in G.nodes():
            node_degree = G.degree(node)
            if node_degree > 0:
                community_connections = {}
                for neighbor in G.neighbors(node):
                    neighbor_community = partition[neighbor]
                    community_connections[neighbor_community] = (
                        community_connections.get(neighbor_community, 0) + 1
                    )
                if len(community_connections) > 1:
                    probs = [count / node_degree for count in community_connections.values()]
                    entropy = -sum(p * np.log2(p) for p in probs if p > 0)
                    participation_entropies.append(entropy)
                    if return_node_level:
                        node_participation_entropies[node] = entropy

        if participation_entropies:
            metrics[f"{metric_type}_participation_entropy"] = np.mean(participation_entropies)
        else:
            metrics[f"{metric_type}_participation_entropy"] = np.nan
    else:
        metrics[f"{metric_type}_participation_entropy"] = np.nan

    if return_node_level:
        metrics["node_level"] = {
            "participation_coefficient": node_participation_coeffs,
            "within_module_degree": node_within_module_degrees,
            "participation_entropy": node_participation_entropies,
        }

    return metrics

def calculate_unified_metrics(results_df):
    """计算跨阈限的统一指标"""
    print("计算跨阈限的统一指标...")

    exclude_cols = ["subject", "state", "threshold"]
    metric_cols = [col for col in results_df.columns if col not in exclude_cols]
    unified_results = []

    for (subj, state), group in results_df.groupby(["subject", "state"]):
        if len(group) < 2:
            continue

        unified_row = {"subject": subj, "state": state}

        for metric in metric_cols:
            values = group[metric].dropna()
            if len(values) > 0:
                unified_row[f"{metric}_mean"] = float(np.mean(values))
                if len(values) > 1:
                    try:
                        sorted_data = sorted(
                            zip(group["threshold"].values[: len(values)], values)
                        )
                        thresholds_sorted, values_sorted = zip(*sorted_data)
                        auc = np.trapz(values_sorted, thresholds_sorted)
                        unified_row[f"{metric}_auc"] = float(auc)
                    except Exception:
                        unified_row[f"{metric}_auc"] = np.nan
                else:
                    unified_row[f"{metric}_auc"] = np.nan
            else:
                unified_row[f"{metric}_mean"] = np.nan
                unified_row[f"{metric}_auc"] = np.nan

        unified_results.append(unified_row)

    unified_df = pd.DataFrame(unified_results)
    print(f"统一指标计算完成，共 {len(unified_df)} 行数据")
    return unified_df
